In [0]:
process_name="pricing_data_transformation_silver"
File_Folder_Name="daily_pricing"
bronze_Layer_Path=f"/Volumes/pricing_analytics/storageac/pricing_analytics/bronze/{File_Folder_Name}/"
bronze_catalog_path=f"pricing_analytics.bronze.{File_Folder_Name}"
silver_Layer_Path=f"/Volumes/pricing_analytics/storageac/pricing_analytics/silver/{File_Folder_Name}/"
silver_catalog_path=f"pricing_analytics.silver.{File_Folder_Name}"
pipeline_control_catalog="pricing_analytics.bronze.pipeline_control_logs"

In [0]:
from pyspark.sql.functions import col, coalesce, to_date, current_timestamp, row_number, current_date
from pyspark.sql.types import LongType, DecimalType
from pyspark.sql.window import Window

In [0]:
bronze_data_df=spark.sql(f"""select * from {bronze_catalog_path} where File_updated_date>(select coalesce(max(process_table_date_time), date('1900-01-01')) from {pipeline_control_catalog} where process_name='{process_name}' and process_status='Completed')""")

In [0]:
strict_schema_df=(bronze_data_df.select(coalesce(to_date('DATE_OF_PRICING', 'dd/MM/yyyy'), current_date()).alias('DATE_OF_PRICING'),
                                col('ROW_ID').cast(LongType()),
                                col('STATE_NAME'),
                                col('MARKET_NAME'),
                                col('PRODUCTGROUP_NAME'),
                                col('PRODUCT_NAME'),
                                col('ORIGIN'),
                                col('ARRIVAL_IN_TONNES').try_cast(DecimalType(12, 2)),
                                col('MINIMUM_PRICE').try_cast(DecimalType(18, 2)),
                                col('MAXIMUM_PRICE').try_cast(DecimalType(18, 2)),
                                col('MODAL_PRICE').try_cast(DecimalType(18, 2)),
                                col('File_updated_date').alias('Bronze_File_Updated_date')
                                )
)

In [0]:
strict_schema_df.createOrReplaceTempView('bronze_data')

In [0]:
# remove duplicates 
clean_df=strict_schema_df.withColumn("rn", row_number().over(Window.partitionBy('DATE_OF_PRICING', 'ROW_ID').orderBy(col('Bronze_File_Updated_date').desc()))).filter(col('rn')==1).drop('rn', 'Bronze_File_Updated_date')

# Handeling nulls based on business conditions 
valid_df=clean_df.filter((col('DATE_OF_PRICING').isNotNull()) & (col('ROW_ID').isNotNull()) & (col('MODAL_PRICE')>0))

quarntine_df=clean_df.filter((col('DATE_OF_PRICING').isNull()) | (col('ROW_ID').isNull()) | (col('MODAL_PRICE')<=0))

valid_df=valid_df.fillna(0, subset=['ARRIVAL_IN_TONNES', 'MINIMUM_PRICE', 'MAXIMUM_PRICE'])\
    .fillna('NA', subset=['MARKET_NAME'])

final_df=valid_df.withColumn('LAKE_HOUSE_INSERTED_DATE', current_timestamp())\
    .withColumn('LAKE_HOUSE_UPDATED_DATE', current_timestamp())


In [0]:
spark.sql(f"""insert into pricing_analytics.bronze.file_process_data_count_log(
    select
    '{process_name}' as process_name,
    '{File_Folder_Name}' as file_name,
    count(*) as Total_count,
    sum(case when DATE_OF_PRICING is not null and ROW_ID is not null and MODAL_PRICE>0 then 1 else 0 end) as Valid_count,
    sum(case when DATE_OF_PRICING is null or ROW_ID is null or MODAL_PRICE<=0 then 1 else 0 end) as Quarntine_count,
    current_timestamp() as process_timestamp
    from bronze_data
)""")

In [0]:
from delta.tables import DeltaTable

delta_df=DeltaTable.forName(spark, silver_catalog_path)

if spark.catalog.tableExists(silver_catalog_path):
    delta_table=DeltaTable.forName(spark, silver_catalog_path)

    delta_table.alias('target').merge(final_df.alias('source'), 'target.DATE_OF_PRICING=source.DATE_OF_PRICING and target.ROW_ID=source.ROW_ID')

else:
    final_df.write.format('delta').mode('overwrite').saveAsTable(silver_catalog_path)
